# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sheby-me/FLYRANK-Onboarding/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
The Rule: A page should be flagged for a refresh if it is stale (hasn't been updated in over 180 days), is slipping in rankings (average position is greater than 10), and has a meaningful search volume (>0). We prioritize the queue by multiplying these boolean flags by the search_volume to rank the biggest opportunities first.

Reason Codes:

stale_slipping_high_vol

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('https://raw.githubusercontent.com/sheby-me/FLYRANK-Onboarding/refs/heads/main/data/raw/content_refresh_anonymized.csv')

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
# Signal 1: Staleness (Days since last update)
# Let's bucket them into <180 days and >180 days
df['is_stale'] = df['days_since_last_update'] > 180
print("--- Signal 1: Staleness ---")
print(df['is_stale'].value_counts(dropna=False))
print("Verdict: CONFIRMED\n")

# Signal 2: Position Slipping (Page 2 or worse, excluding 0)
# Positions 11+ are usually Page 2+ in search results
df['is_slipping'] = (df['avg_position'] > 10) & (df['avg_position'] != 0)
print("--- Signal 2: Slipping Position ---")
print(df['is_slipping'].value_counts(dropna=False))
print("Verdict: CONFIRMED")

Loaded 30000 rows and 44 columns.
--- Signal 1: Staleness ---
is_stale
False    29826
True       174
Name: count, dtype: int64
Verdict: CONFIRMED

--- Signal 2: Slipping Position ---
is_slipping
True     15812
False    14188
Name: count, dtype: int64
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Calculate the score (Boolean logic multiplied by volume for ranking weight)
df['action_score'] = (
    (df['days_since_last_update'] > 180).astype(int) *
    ((df['avg_position'] > 10) & (df['avg_position'] != 0)).astype(int) *
    df['search_volume']
)

# 2. Assign Reason Codes
df['reason_code'] = np.where(df['action_score'] > 0, 'stale_slipping_high_vol', 'none')

# 3. Sort by score descending to build the ranked queue
ranked_queue = df.sort_values(by='action_score', ascending=False).copy()

# 4. Export to the requested output path
import os
os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Ranked queue saved successfully!")

Ranked queue saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_20 = ranked_queue[ranked_queue['action_score'] > 0].head(10)

print("=== TOP 10 PICKS FOR REFRESH ===")
for index, row in top_20.iterrows():
    print(f"Content ID: {row['content_id']}")
    print(f"Action: Refresh Content")
    print(f"Reason Code: {row['reason_code']}")
    print(f"Metrics -> Volume: {row['search_volume']}, Pos: {row['avg_position']}, Days Stale: {row['days_since_last_update']}")
    print("-" * 40)

=== TOP 10 PICKS FOR REFRESH ===
Content ID: content_bbca724138f2
Action: Refresh Content
Reason Code: stale_slipping_high_vol
Metrics -> Volume: 1600.0, Pos: 12.1, Days Stale: 236
----------------------------------------
Content ID: content_7a888d3d99c8
Action: Refresh Content
Reason Code: stale_slipping_high_vol
Metrics -> Volume: 90.0, Pos: 67.6, Days Stale: 313
----------------------------------------
Content ID: content_dd413158df3c
Action: Refresh Content
Reason Code: stale_slipping_high_vol
Metrics -> Volume: 20.0, Pos: 46.1, Days Stale: 305
----------------------------------------
Content ID: content_026a1e2a82fd
Action: Refresh Content
Reason Code: stale_slipping_high_vol
Metrics -> Volume: 10.0, Pos: 34.0, Days Stale: 305
----------------------------------------
Content ID: content_f2b4acf220d9
Action: Refresh Content
Reason Code: stale_slipping_high_vol
Metrics -> Volume: 10.0, Pos: 17.5, Days Stale: 305
----------------------------------------
Content ID: content_ab18b5811c

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Leakage Check:
I have confirmed that no future window metrics or derivative labels (such as trend_direction or trend_pct) were used in the calculation of action_score. The baseline relies strictly on current state facts (days_since_last_update, avg_position, and search_volume).

Weak Pick Identification:
A logic flaw in this baseline is that it heavily biases toward search_volume as the tie-breaker/multiplier. A piece of content might have a massive search volume but an incredibly low ctr (e.g., 0.01%), meaning the content type might be fundamentally wrong for the keyword (e.g., an informational article trying to rank for a transactional keyword). Ranking purely by volume might waste editorial effort on impossible keywords.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.